<a href="https://colab.research.google.com/github/ramanji567/trends_pulse_task2_data_processing/blob/main/task1_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import requests
import time
import json
from datetime import datetime

headers = {
    "User-Agent":"TrendPluse/1.0"
}
# Corrected: Assign data directly to data_1
categories = {
    "technology":["AI","software","tech","code","computer","data","cloud","API","GPU","LLM"],
    "worldnews":["war","government","country","president","election","climate","attack","global"],
    "sports":["NFL","NBA","FIFA","sport","game","team","player","league","championship"],
    "science":["research","study","space","physics","biology","discovery","NASA","genome"],
    "entertainment":["movie","film","music","Netflix","game","book","show","award","streaming"]

}
def get_category (title):
  title = title.lower()
  for category_name,keywords in categories.items(): # Renamed 'keyword' to 'category_name' to avoid conflict
    for word in keywords:
      if word in title:
        return category_name # Return category_name here
  return "other"

def get_story_id():
  try:
    url = f"https://hacker-news.firebaseio.com/v0/topstories.json"
    response = requests.get(url,headers = headers)
    return response.json()
    # print(data) # Removed unreachable and undefined 'print(data)'
  except:
    print("failed to fetch")
    return [] # Return an empty list on failure

def get_story(story_id):

   try:
      url = f"https://hacker-news.firebaseio.com/v0/item/{story_id}.json" # Corrected to use the passed story_id
      response = requests.get(url,headers=headers)
      return response.json()
   except:
     print(f"failed to fetch story {story_id}") # Added story_id to error message
     return None # Return None on failure
def main():
  story_ids = get_story_id()

  result = {
      "technology":[],
      "worldnews":[],
      "sports":[],
      "science":[],
      "entertainment":[]
  }

  for category in result:
    count =0

    for story_id in story_ids:
      if count >=25:
        break

      story = get_story(story_id)
      if not story or "title" not in story:
        continue
      category_name = get_category(story["title"]) # Corrected to call get_category with the story title
      if category_name == category:
        data ={
            "post_id":story.get("id"),
            "title":story.get("title"),
            "category":category_name,
            "score":story.get("score"),
            "num_comments":story.get("descendants"),
            "author":story.get("by"),
            "collected_at":datetime.now().isoformat()
        }

        result[category].append(data)
        count += 1 # Increment count when a story is added to a category
    #time.sleep(1) # Reduced sleep time to 1 second for faster execution

  if not os.path.exists("data"):
     os.makedirs("data")
  filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.json"

  with open(filename,"w")as f:
      json.dump(result,f,indent=2)
  print(f"saved file: {filename}") # Corrected print statement

if __name__=="__main__":
  main()

saved file: data/trends_20260409.json


In [84]:
import json
import pandas as pd

# Load the JSON data
with open("data/trends_20260409.json", 'r') as f:
    data = json.load(f)

# Initialize an empty list to store DataFrames for each category
all_category_dfs = []

# Iterate through each category and its stories to create DataFrames
for category_name, stories in data.items():
    if stories: # Only create a DataFrame if there are stories in the category
        category_df = pd.DataFrame(stories)
        all_category_dfs.append(category_df)

# Concatenate all category DataFrames into a single DataFrame 'df'
# If all_category_dfs is empty, df will be an empty DataFrame
if all_category_dfs:
    df = pd.concat(all_category_dfs, ignore_index=True)
else:
    df = pd.DataFrame() # Create an empty DataFrame if no data was processed

# Display the DataFrame head and info to verify
print(f"Successfully loaded {len(df)} stories.")
df

Successfully loaded 100 stories.


,post_id,title,category,score,num_comments,author,collected_at
0,47695012,USB for Software Developers: An introduction t...,technology,219,29,WerWolv,2026-04-09T04:00:57.898220
1,47687273,Git commands I run before reading any code,technology,1868,395,grepsedawk,2026-04-09T04:00:58.415687
2,47689237,US cities are axing Flock Safety surveillance ...,technology,658,385,giuliomagnifico,2026-04-09T04:00:59.610521
3,47674606,Show HN: Unicode Steganography,technology,47,14,PatrickVuscan,2026-04-09T04:01:00.521819
4,47656518,Your File System Is Already A Graph Database,technology,164,71,alxndr,2026-04-09T04:01:00.617421
...,...,...,...,...,...,...,...
95,47690950,Show HN: BAREmail ʕ·ᴥ·ʔ – minimalist Gmail cli...,entertainment,43,40,Virgo_matt,2026-04-09T04:03:31.469581
96,47636579,Show HN: Anos – a hand-written ~100KiB microke...,entertainment,113,31,noone_youknow,2026-04-09T04:03:31.681840
97,47655408,Show HN: I built a tiny LLM to demystify how l...,entertainment,900,134,armanified,2026-04-09T04:03:32.272245
98,47673394,"Show HN: Stop paying for Dropbox/Google Drive,...",entertainment,245,200,Zm44,2026-04-09T04:03:32.508951


In [68]:
df .info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   post_id       100 non-null    int64 
 1   title         100 non-null    object
 2   category      100 non-null    object
 3   score         100 non-null    int64 
 4   num_comments  100 non-null    int64 
 5   author        100 non-null    object
 6   collected_at  100 non-null    object
dtypes: int64(3), object(4)
memory usage: 5.6+ KB


In [69]:
df.isnull().sum()

,0
post_id,0
title,0
category,0
score,0
num_comments,0
author,0
collected_at,0


In [70]:
df[df.duplicated(keep=False)].sort_values("post_id")

,post_id,title,category,score,num_comments,author,collected_at


In [71]:
trend_col=["title","category","author"]
for i in trend_col:
  unique_values = df[i].unique()
  print(f"{len(unique_values)}")
  print(sorted(unique_values))

100
['A case study in testing with 100+ Claude agents in parallel', 'A database of analog cameras that can be 3D printed', 'AMD AI director says Claude Code is becoming dumber and lazier since update', 'Adobe modifies hosts file to detect whether Creative Cloud is installed', 'Amazon rewards loyal Kindle devotees by closing the book on old e-readers', "Anthropic's Restraint Is a Terrifying Warning Sign", 'Apollo Guidance Computer restoration videos', 'Battle for Wesnoth: open-source, turn-based strategy game', 'Breaking the console: a brief history of video game security', 'Bun: cgroup-aware AvailableParallelism / HardwareConcurrency on Linux', 'Case study: recovery of a corrupted 12 TB multi-device pool', 'Cloudflare targets 2029 for full post-quantum security', 'Computational Physics (2nd Edition) (2025)', "Databricks co-founder wins prestigious ACM award, says 'AGI is here already'", 'Digital Hopes, Real Power: How the Arab Spring Fueled a Global Surveillance Boom', 'Dropping Cloudf

In [72]:
quality = df[df["score"]<5]

In [73]:
df.drop (df[df["score"] < 5].index,inplace=True)

In [74]:
df

,post_id,title,category,score,num_comments,author,collected_at
0,47695012,USB for Software Developers: An introduction t...,technology,219,29,WerWolv,2026-04-09T04:00:57.898220
1,47687273,Git commands I run before reading any code,technology,1868,395,grepsedawk,2026-04-09T04:00:58.415687
2,47689237,US cities are axing Flock Safety surveillance ...,technology,658,385,giuliomagnifico,2026-04-09T04:00:59.610521
3,47674606,Show HN: Unicode Steganography,technology,47,14,PatrickVuscan,2026-04-09T04:01:00.521819
4,47656518,Your File System Is Already A Graph Database,technology,164,71,alxndr,2026-04-09T04:01:00.617421
...,...,...,...,...,...,...,...
95,47690950,Show HN: BAREmail ʕ·ᴥ·ʔ – minimalist Gmail cli...,entertainment,43,40,Virgo_matt,2026-04-09T04:03:31.469581
96,47636579,Show HN: Anos – a hand-written ~100KiB microke...,entertainment,113,31,noone_youknow,2026-04-09T04:03:31.681840
97,47655408,Show HN: I built a tiny LLM to demystify how l...,entertainment,900,134,armanified,2026-04-09T04:03:32.272245
98,47673394,"Show HN: Stop paying for Dropbox/Google Drive,...",entertainment,245,200,Zm44,2026-04-09T04:03:32.508951


In [75]:
df

,post_id,title,category,score,num_comments,author,collected_at
0,47695012,USB for Software Developers: An introduction t...,technology,219,29,WerWolv,2026-04-09T04:00:57.898220
1,47687273,Git commands I run before reading any code,technology,1868,395,grepsedawk,2026-04-09T04:00:58.415687
2,47689237,US cities are axing Flock Safety surveillance ...,technology,658,385,giuliomagnifico,2026-04-09T04:00:59.610521
3,47674606,Show HN: Unicode Steganography,technology,47,14,PatrickVuscan,2026-04-09T04:01:00.521819
4,47656518,Your File System Is Already A Graph Database,technology,164,71,alxndr,2026-04-09T04:01:00.617421
...,...,...,...,...,...,...,...
95,47690950,Show HN: BAREmail ʕ·ᴥ·ʔ – minimalist Gmail cli...,entertainment,43,40,Virgo_matt,2026-04-09T04:03:31.469581
96,47636579,Show HN: Anos – a hand-written ~100KiB microke...,entertainment,113,31,noone_youknow,2026-04-09T04:03:31.681840
97,47655408,Show HN: I built a tiny LLM to demystify how l...,entertainment,900,134,armanified,2026-04-09T04:03:32.272245
98,47673394,"Show HN: Stop paying for Dropbox/Google Drive,...",entertainment,245,200,Zm44,2026-04-09T04:03:32.508951


In [77]:
df["title"]=df['title'].str.strip()
print(df["title"].value_counts())


title
USB for Software Developers: An introduction to writing userspace USB drivers       1
Git commands I run before reading any code                                          1
US cities are axing Flock Safety surveillance technology                            1
Show HN: Unicode Steganography                                                      1
Your File System Is Already A Graph Database                                        1
                                                                                   ..
Show HN: BAREmail ʕ·ᴥ·ʔ – minimalist Gmail client for bad WiFi                      1
Show HN: Anos – a hand-written ~100KiB microkernel for x86-64 and RISC-V            1
Show HN: I built a tiny LLM to demystify how language models work                   1
Show HN: Stop paying for Dropbox/Google Drive, use your own S3 bucket instead       1
Show HN: Finalrun – Spec-driven testing using English and vision for mobile apps    1
Name: count, Length: 99, dtype: int64


In [79]:
print(f"loaded 100 stories from data/trends_20260409.json")

loaded 100 stories from data/trends_20260409.json


In [83]:
print(f"after 0 duplicates are there ")
print(f" after 0 null values are there")
print(f"reomoving low scores: 1")
print(f"saved {len(df)} rows to data/trends_clean.csv")

after 0 duplicates are there 
 after 0 null values are there
reomoving low scores: 1
saved 99 rows to data/trends_clean.csv


In [88]:
import csv
import os
from datetime import datetime
os.makedirs("data",exist_ok=True)

filename = f"data/trends_{datetime.now().strftime('%Y%m%d')}.csv"
df.to_csv(filename,index=False,encoding="utf-8")
print("csv saved:",filename)

csv saved: data/trends_20260409.csv


In [97]:
import pandas as pd
df = pd.read_csv("data/trends_20260409.csv")
df

,post_id,title,category,score,num_comments,author,collected_at
0,47695012,USB for Software Developers: An introduction t...,technology,219,29,WerWolv,2026-04-09T04:00:57.898220
1,47687273,Git commands I run before reading any code,technology,1868,395,grepsedawk,2026-04-09T04:00:58.415687
2,47689237,US cities are axing Flock Safety surveillance ...,technology,658,385,giuliomagnifico,2026-04-09T04:00:59.610521
3,47674606,Show HN: Unicode Steganography,technology,47,14,PatrickVuscan,2026-04-09T04:01:00.521819
4,47656518,Your File System Is Already A Graph Database,technology,164,71,alxndr,2026-04-09T04:01:00.617421
...,...,...,...,...,...,...,...
95,47690950,Show HN: BAREmail ʕ·ᴥ·ʔ – minimalist Gmail cli...,entertainment,43,40,Virgo_matt,2026-04-09T04:03:31.469581
96,47636579,Show HN: Anos – a hand-written ~100KiB microke...,entertainment,113,31,noone_youknow,2026-04-09T04:03:31.681840
97,47655408,Show HN: I built a tiny LLM to demystify how l...,entertainment,900,134,armanified,2026-04-09T04:03:32.272245
98,47673394,"Show HN: Stop paying for Dropbox/Google Drive,...",entertainment,245,200,Zm44,2026-04-09T04:03:32.508951


In [98]:
df[df["score"]<5]

,post_id,title,category,score,num_comments,author,collected_at
62,47672190,Ray – OpenSubtitles upcoming auto translating/...,sports,3,0,nolok,2026-04-09T04:02:34.757081


In [104]:
df.drop(df[df["score"]<5].index,inplace=True)
df

,post_id,title,category,score,num_comments,author,collected_at
0,47695012,USB for Software Developers: An introduction t...,technology,219,29,WerWolv,2026-04-09T04:00:57.898220
1,47687273,Git commands I run before reading any code,technology,1868,395,grepsedawk,2026-04-09T04:00:58.415687
2,47689237,US cities are axing Flock Safety surveillance ...,technology,658,385,giuliomagnifico,2026-04-09T04:00:59.610521
3,47674606,Show HN: Unicode Steganography,technology,47,14,PatrickVuscan,2026-04-09T04:01:00.521819
4,47656518,Your File System Is Already A Graph Database,technology,164,71,alxndr,2026-04-09T04:01:00.617421
...,...,...,...,...,...,...,...
95,47690950,Show HN: BAREmail ʕ·ᴥ·ʔ – minimalist Gmail cli...,entertainment,43,40,Virgo_matt,2026-04-09T04:03:31.469581
96,47636579,Show HN: Anos – a hand-written ~100KiB microke...,entertainment,113,31,noone_youknow,2026-04-09T04:03:31.681840
97,47655408,Show HN: I built a tiny LLM to demystify how l...,entertainment,900,134,armanified,2026-04-09T04:03:32.272245
98,47673394,"Show HN: Stop paying for Dropbox/Google Drive,...",entertainment,245,200,Zm44,2026-04-09T04:03:32.508951


In [105]:
import pandas as pd
data = pd.DataFrame(df).to_csv("trends.202260409.csv",index=False)
